# 01 — Exploratory Data Analysis

Analyse the merged prompt dataset: class distribution, text lengths, and sample examples per threat class.

**Run after:** `python data/download.py && python data/prepare.py`

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

splits = {
    s: [json.loads(line) for line in (Path("data/processed") / f"{s}.jsonl").read_text().splitlines()]
    for s in ["train", "val", "test"]
}
dfs = {s: pd.DataFrame(rows) for s, rows in splits.items()}
print({s: len(df) for s, df in dfs.items()})

## Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (split, df) in zip(axes, dfs.items()):
    df["label"].value_counts().sort_index().plot(kind="bar", ax=ax, title=split)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("notebooks/eda_class_dist.png", dpi=150)
plt.show()

## Text length distribution

In [ ]:
df_all = pd.concat(dfs.values(), ignore_index=True)
df_all["n_chars"] = df_all["text"].str.len()
df_all["n_words"] = df_all["text"].str.split().str.len()
display(df_all.groupby("label")[["n_chars", "n_words"]].describe().round(1))

## Sample examples per class

In [ ]:
for label in sorted(dfs["train"]["label"].unique()):
    sample = dfs["train"][dfs["train"]["label"] == label].sample(2, random_state=0)
    print(f"\n=== {label} ===")
    for _, row in sample.iterrows():
        print(f"  {row['text'][:120]!r}")